In [ ]:
import ast
import concurrent.futures
import glob
import itertools
import os
import pickle
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

import dask
import dask.dataframe as dd
import dask_ml.cluster as dask_cluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar

from concurrent.futures import ThreadPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
def read_csv_helper(fpath):
    splits = fpath.split("=")
    cutoff = splits[1].split(".")[0]
    #fips = splits[1].split("_")[0]
    df = pd.read_csv(fpath)
    
    old_cols = list(df.columns)
    
    #df["fips"] = int(fips)
    df["cutoff"] = int(cutoff)
    
    df = df[["cutoff"] + old_cols]
    
    return df

In [ ]:
# Load pre-computed leaf/tree size data
date_combined_df = pd.read_csv('TLGRF_leaf_and_tree_size_by_date.csv', index_col=0, parse_dates=True)
date_combined_df.index = pd.to_datetime(date_combined_df.index)
print('loaded TLGRF_leaf_and_tree_size_by_date.csv:', date_combined_df.shape)
print(date_combined_df.head())


In [ ]:
pass  # cutoff_to_date skipped


In [ ]:
date_combined_df

### Analyse Leafs by Cutoff

In [ ]:
# Use pre-computed aggregated values directly
mean_leaf_size_by_cutoff = date_combined_df['Mean Leaf Size']
median_leaf_size_by_cutoff = date_combined_df['Median Leaf Size']
mean_tree_size_by_cutoff = date_combined_df['Mean Tree Size']
mean_sample_size_by_cutoff = date_combined_df['Mean Tree Size']
median_sample_size_by_cutoff = date_combined_df['Median Tree Size']
median_tree_size_by_cutoff = date_combined_df['Median Tree Size']
mean_ratio_by_cutoff = date_combined_df['Mean Ratio']
median_ratio_by_cutoff = date_combined_df['Median Ratio']


In [ ]:
plt.figure(figsize=(20,10))
plt.plot(mean_leaf_size_by_cutoff, label="Mean Leaf Size")
plt.plot(median_leaf_size_by_cutoff, label="Median Leaf Size")

#plt.plot(mean_sample_size_by_cutoff, label="Mean Training Sample Size")
#plt.plot(median_sample_size_by_cutoff, label="Median Training Sample Size")
plt.xlim(pd.to_datetime("2020-03-15"), pd.to_datetime("2023-01-01"))

plt.xlabel("Date")
plt.ylabel("Number of Samples")
plt.title("Mean and Median TLGRF Test Sample's Leaf Size")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(20,10))
plt.plot(mean_ratio_by_cutoff, label="Mean Proportion of Leaf to Tree")
plt.plot(median_ratio_by_cutoff, label="Median Proportion of Leaf to Tree")
plt.xlim(pd.to_datetime("2020-03-15"), pd.to_datetime("2023-01-01"))

plt.xlabel("Date")
plt.ylabel("Number of Samples")
plt.title("Proportion of TLGRF Test Sample's Leaf Size to Tree")
plt.legend()
plt.show()

### Save Results

In [ ]:
TLGRF_leaf_and_tree_size_by_date = pd.DataFrame()
TLGRF_leaf_and_tree_size_by_date["Mean Leaf Size"] = mean_leaf_size_by_cutoff
TLGRF_leaf_and_tree_size_by_date["Median Leaf Size"] = median_leaf_size_by_cutoff

TLGRF_leaf_and_tree_size_by_date["Mean Tree Size"] = mean_sample_size_by_cutoff
TLGRF_leaf_and_tree_size_by_date["Median Tree Size"] = median_sample_size_by_cutoff

TLGRF_leaf_and_tree_size_by_date["Mean Ratio"] = mean_ratio_by_cutoff
TLGRF_leaf_and_tree_size_by_date["Median Ratio"] = median_ratio_by_cutoff

TLGRF_leaf_and_tree_size_by_date

In [ ]:
TLGRF_leaf_and_tree_size_by_date.to_csv("TLGRF_leaf_and_tree_size_by_date.csv", index=True)